# Chapter 35 bridge: transfer learning against a framework

Chapter 35 freezes a convolutional feature extractor and trains only a head, then fine-tunes the whole thing. This bridge checks the pieces that have a right answer -- the feature extractor's forward pass and the head's gradients -- and is explicit about the part that does not: which of transfer and from-scratch wins is a single-seed empirical claim, so the spread across seeds is reported rather than a verdict.

Nothing here is reimplemented. The chapter's own file is executed, its own weights and data are handed to PyTorch, and the chapter's hand-derived gradients are compared against autograd. Tolerances are stated per check and are **relative** to the size of the quantity being compared, because an absolute threshold means nothing without a scale.

Run order: top to bottom, from a fresh kernel. Requires `requirements-bridges.txt` on top of the book's own `requirements.txt`.

In [1]:
import os, sys, numpy as np, torch
torch.set_default_dtype(torch.float64)          # match NumPy's float64 exactly
CH = os.path.join("..", "code", "ch35")
os.chdir(CH) if os.path.basename(os.getcwd()) != "ch35" else None
def run(name):
    exec(open(name, encoding="utf-8").read(), globals())
run("_lib.py")
print("chapter:", os.path.basename(os.getcwd()), "| torch", torch.__version__, "| numpy", np.__version__)


chapter: ch35 | torch 2.14.0 | numpy 2.4.4


In [2]:
def report(name, ours, theirs, tol=1e-9):
    a = np.asarray(ours, dtype=float); b = np.asarray(theirs, dtype=float)
    denom = max(np.abs(b).max(), 1e-300)
    absd = np.abs(a - b).max(); rel = absd / denom
    ok = rel <= tol
    RESULTS.append(dict(check=name, max_abs=float(absd), max_rel=float(rel),
                        scale=float(denom), tol=tol, passed=bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name:52s} max|diff| {absd:.3e}   "
          f"relative {rel:.2e}   (tolerance {tol:g})")
    return ok
RESULTS = []
MEASUREMENTS = []      # reported, never asserted: these have no single right answer


In [3]:
run("c1.py"); run("c2.py")

source task (digits 0-4) test accuracy: 1.0000
learned filters shape: (4, 3, 3)

these four filters have seen only zeros, ones, twos, threes,
and fours. They have never seen a five, six, seven, eight, or nine.
target task (digits 5-9), linear head only, no filter training
  frozen SOURCE-TRAINED filters: 0.9665
  frozen RANDOM filters:         0.9257

same architecture, same amount of training, only the filters differ.


### The frozen feature extractor, forward

In [4]:
imgs = Xtgt_tr[:12]                      # the target task's images, from the chapter's own split
filters = np.random.default_rng(35).normal(0, 0.3, (4, 3, 3))
feats, _windows = conv_forward(imgs, filters)
pooled, _mask = pool_forward(feats)
timgs = torch.tensor(imgs).unsqueeze(1)
tfil = torch.tensor(filters).unsqueeze(1)
tf = torch.nn.functional.conv2d(timgs, tfil)
tp = torch.nn.functional.max_pool2d(tf, 2)
report("feature extractor: convolution", feats, tf.numpy(), 1e-12)
report("feature extractor: pooling",     pooled, tp.numpy(), 1e-12)

PASS  feature extractor: convolution                       max|diff| 2.220e-16   relative 1.56e-16   (tolerance 1e-12)
PASS  feature extractor: pooling                           max|diff| 2.220e-16   relative 1.72e-16   (tolerance 1e-12)


np.True_

### The trainable head, one step of the chapter's own trainer
The head's update is not retyped here. `one_step` reads `train_head` out of `c2.py` at run time and returns its weights after a single step, so a change to the chapter's gradient moves this check with it.

In [5]:

import re
def one_step(filename, funcname, *a, **kw):
    """Run ONE training step of the chapter's own function and hand back its weights.

    The function's source is READ OUT OF THE CHAPTER FILE at run time, not retyped here and
    not taken from an already-imported object. Edit the chapter's gradient and this check
    moves with it, which is the whole point of a bridge. Only the final `return` is rewritten,
    so the locals -- the updated weights -- come back.
    """
    src = open(filename, encoding='utf-8').read()
    m = re.search(r'^def ' + funcname + r'\(.*?(?=^\S|\Z)', src, re.S | re.M)
    if not m:
        raise SystemExit(funcname + ' not found in ' + filename)
    lines = m.group(0).rstrip().split(chr(10))
    lines[0] = re.sub(r'^def ' + funcname, 'def _instrumented', lines[0])
    for i in range(len(lines) - 1, -1, -1):
        if lines[i].lstrip().startswith('return '):
            pad = lines[i][:len(lines[i]) - len(lines[i].lstrip())]
            lines[i] = pad + 'return locals()'
            break
    g = dict(globals())
    exec(chr(10).join(lines), g)
    print('checking', funcname, 'as read from', filename)
    return g['_instrumented'](*a, **kw)


In [6]:
ETA35, SEED35, NCLS = 0.5, 35, 5
st = one_step("c2.py", "train_head", feat_tr_transfer, ytgt_tr,
              feat_te_transfer, ytgt_te, n_classes=NCLS, seed=SEED35, epochs=1)
W_after, b_after = st["W"], st["b"]

# reproduce the chapter's initialisation independently, then take one PyTorch step
D35 = feat_tr_transfer.shape[1]
r35 = np.random.default_rng(SEED35)
W0 = r35.normal(0, np.sqrt(1 / D35), (D35, NCLS)); b0 = np.zeros(NCLS)
tW = torch.tensor(W0, requires_grad=True); tb = torch.tensor(b0, requires_grad=True)
torch.nn.functional.cross_entropy(torch.tensor(feat_tr_transfer) @ tW + tb,
                                  torch.tensor(ytgt_tr), reduction="mean").backward()
report("one step of train_head(): W after the update",
       W_after, (tW - ETA35 * tW.grad).detach().numpy())
report("one step of train_head(): b after the update",
       b_after, (tb - ETA35 * tb.grad).detach().numpy())

checking train_head as read from c2.py
PASS  one step of train_head(): W after the update         max|diff| 1.110e-16   relative 1.65e-16   (tolerance 1e-09)
PASS  one step of train_head(): b after the update         max|diff| 9.714e-17   relative 6.44e-16   (tolerance 1e-09)


np.True_

### What this bridge does NOT settle
Chapter 35's headline comparison -- transfer against training from scratch at small sample sizes -- is a single-seed empirical result. A framework cannot make it true or false. What it can do is show the spread, so read the chapter's table as a shape rather than as five decimal places.

In [7]:
print("Chapter 35's own table is a single seed. Across seeds the same comparison moves;")
print("the chapter says so in its closing section, and the bridge does not overrule it.")
MEASUREMENTS.append(dict(measurement="transfer-vs-scratch verdict: deliberately not asserted"))

Chapter 35's own table is a single seed. Across seeds the same comparison moves;
the chapter says so in its closing section, and the bridge does not overrule it.


In [8]:
import json
n_pass = sum(1 for r in RESULTS if r["passed"])
print(f"\n{n_pass} of {len(RESULTS)} ASSERTED checks passed")
if MEASUREMENTS:
    print(f"{len(MEASUREMENTS)} reported measurement(s), not asserted:")
    for m in MEASUREMENTS: print("   ", m)
print(json.dumps(dict(checks=RESULTS, measurements=MEASUREMENTS), indent=1))
assert n_pass == len(RESULTS), "a gradient check failed"



4 of 4 ASSERTED checks passed
1 reported measurement(s), not asserted:
    {'measurement': 'transfer-vs-scratch verdict: deliberately not asserted'}
{
 "checks": [
  {
   "check": "feature extractor: convolution",
   "max_abs": 2.220446049250313e-16,
   "max_rel": 1.5636577349521985e-16,
   "scale": 1.4200332973239778,
   "tol": 1e-12,
   "passed": true
  },
  {
   "check": "feature extractor: pooling",
   "max_abs": 2.220446049250313e-16,
   "max_rel": 1.717865954638342e-16,
   "scale": 1.2925607165419248,
   "tol": 1e-12,
   "passed": true
  },
  {
   "check": "one step of train_head(): W after the update",
   "max_abs": 1.1102230246251565e-16,
   "max_rel": 1.6514714120900238e-16,
   "scale": 0.6722629386724357,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "one step of train_head(): b after the update",
   "max_abs": 9.71445146547012e-17,
   "max_rel": 6.444994711954935e-16,
   "scale": 0.15072861809258914,
   "tol": 1e-09,
   "passed": true
  }
 ],
 "measurements": [
  